First of all, set the 'CODE_DIR' to where the model code are saved. This will change current working directory and print for checking. Afterwards, we import all required modules.

In [1]:
import os

# Save the current PATH
original_path = os.environ['PATH']

# Set CUDA 12.5 environment variables, appending the original PATH explicitly
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.5'
os.environ['PATH'] = f"/usr/local/cuda-12.5/bin:{original_path}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-12.5/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

#!rm -rf /home/ids/yuhe/.cache/torch_extensions

CODE_DIR = '/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/pSp_encoder_constructive'

import os
os.chdir(f'{CODE_DIR}')

notebook_path = os.getcwd()
print('Current working directory is:', '\n', notebook_path) 

Current working directory is: 
 /home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/pSp_encoder_constructive


In [2]:

from argparse import Namespace
import time
import sys
import pprint
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(".")
sys.path.append("..")

# from datasets import augmentations
from utils.common import tensor2im, log_input_image
# from models.psp import pSp

from notebooks.def_funcs import load_sparsity_model, load_ema_model, evaluate_model, transform_images_to_batch, load_folder_images, \
    show_latent_map, visulize_singleImg_paired2, visulize_singleImg_paired3, visulize_singleImg_paired4, visulize_singleImg_paired5, evaluate_learn_scaled_st, evaluate_scaled_st
# %load_ext autoreload
# %autoreload 2

## Parameters setting

Fell free to change the golobal parameters for all experiments

In [3]:
EXPERIMENT_PARMS = {
        # "model_orn_path": "/home/ids/yuhe/Projects/CA_with_GAN/3_code_styleGAN/pretrained_models/e4e_models/e4e_ffhq_encode.pt",
        #"model1_path": "../results/csmlp_sparsity/mlp3D/nodim/checkpoints/iteration_100000.pt",
        "model1_path": "./results/csmlp_ffhq_glasses/mlp3D/nodim/checkpoints/iteration_130000.pt",
        "images_bg_path" : "/home/ids/yuhe/Projects/CA_with_GAN/2_data/styleGAN/ffhq_cs/test_bg", 
        "images_t_path" : "/home/ids/yuhe/Projects/CA_with_GAN/2_data/styleGAN/ffhq_cs/test_glass", 
        "model_output_size" : 1024,
        "transform": transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])]) 
}
model1_path = EXPERIMENT_PARMS['model1_path']
#model_weight3_path = EXPERIMENT_PARMS['model_weight3_path']

image_bg_paths = EXPERIMENT_PARMS["images_bg_path"]
image_t_paths = EXPERIMENT_PARMS["images_t_path"]

transform = EXPERIMENT_PARMS['transform']
device = 'cuda'

In [4]:
def test_model_on_latents(output_latents_1, model, device):
    """
    Test the model on latent representations and print logits & probabilities with 4 decimal places.

    Args:
        output_latents_1 (dict): Contains 'latent_bg_c' and 'latent_t_c' tensors.
        model (torch.nn.Module): Trained classification model.
        device (torch.device): CPU or GPU for computation.
    """

    # Assume output_latents_1 contains your CUDA tensors
    latent_bg_c = output_latents_1['latent_bg_c']  # Shape: (4, 18, 512)
    latent_t_c = output_latents_1['latent_t_c']  # Shape: (4, 18, 512)

    # Ensure model is in evaluation mode
    model.eval()

    # ✅ Step 1: Flatten the latents (4, 18, 512) → (4, 9216)
    latent_bg_c = latent_bg_c.view(latent_bg_c.shape[0], -1)  # (4, 9216)
    latent_t_c = latent_t_c.view(latent_t_c.shape[0], -1)  # (4, 9216)

    # # ✅ Step 2: Move latents to the same device as the model
    # latent_bg_c = latent_bg_c.to(device)
    # latent_t_c = latent_t_c.to(device)

    # ✅ Step 3: Forward pass through the model
    with torch.no_grad():
        logits_bg = model(latent_bg_c)  # Raw outputs (logits) for background latents
        logits_t = model(latent_t_c)  # Raw outputs (logits) for target latents

    # ✅ Step 4: Convert logits to probabilities (if using BCEWithLogitsLoss)
    probs_bg = torch.sigmoid(logits_bg)
    probs_t = torch.sigmoid(logits_t)

    # ✅ Print results with 4 decimal places
    print("Background Logits:", [f"{x:.4f}" for x in logits_bg.squeeze().tolist()])
    print("Target Logits:", [f"{x:.4f}" for x in logits_t.squeeze().tolist()])
    print("Background Probabilities:", [f"{x:.4f}" for x in probs_bg.squeeze().tolist()])
    print("Target Probabilities:", [f"{x:.4f}" for x in probs_t.squeeze().tolist()])


### Set parameters

### Load pretrained pSp model

In [5]:
pSp_net, csmlp_net, opts = load_sparsity_model(model1_path, device=device)

Loading trained checkpoint from path: ./results/csmlp_ffhq_glasses/mlp3D/nodim/checkpoints/iteration_130000.pt
training_step:  130000
Loading pSp from checkpoint: ../pretrained_models/pSp_models/psp_ffhq_encode.pt
Loading trained checkpoint from path: ./results/csmlp_ffhq_glasses/mlp3D/nodim/checkpoints/iteration_130000.pt
import models from mlp3D.py
Loading csmlp from path: results/csmlp_sparsity/mlp3D/nodim/


### Perform encode and decode for inversion

In [6]:
# batched_input shape: bs x 3 x w x h
# avaliable seed for ages:   815  6765 7352  9936  476 4432
# Generate a random integer  
# smile seed 53

import random
random_seed = random.randint(0, 10000)  # Using the range of a typical 32-bit signed integer
print('seed: ', random_seed)  
# random_seed = 374
# image_bg_paths = 'notebooks/gender/male'
# image_t_paths = 'notebooks/gender/female'
%matplotlib inline
images_bg = load_folder_images(image_bg_paths, num_images=10, seed=random_seed)
images_t= load_folder_images(image_t_paths, num_images=10, seed=random_seed)

input_images_bg = transform_images_to_batch(images_bg, transform).to(device).float()
input_images_t = transform_images_to_batch(images_t, transform).to(device).float()

# custom_indices = torch.tensor([3, 0, 1, 2])  # Example custom order for batch size of 4
# input_images_bg = input_images_bg[custom_indices]

output_images_1, output_latents_1 = evaluate_model(csmlp_net, pSp_net, input_images_bg, input_images_t, opts)


seed:  7969


In [7]:
model_path = "training_cls/results/sample_cls/2layer_lr0.0001_bs4/checkpoints/model_epoch_10.pth"
from training_cls.cls_models import CustomLatentClassifier
model = CustomLatentClassifier(input_dim=18*512, num_layers=2, hidden_dim=None).to(device)
# Load weights
print(f"Loaded model from {model_path}")
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()  # Set to evaluation mode
test_model_on_latents(output_latents_1, model, device)

Loaded model from training_cls/results/sample_cls/2layer_lr0.0001_bs4/checkpoints/model_epoch_10.pth
Background Logits: ['-9.0016', '-3.5901', '-10.3554', '-5.6512', '-5.6621', '-5.2857', '-6.0534', '-7.2377', '-4.5239', '-7.4189']
Target Logits: ['4.9717', '4.1590', '7.0104', '9.3728', '2.9523', '3.1670', '3.9125', '7.9452', '2.9300', '3.0998']
Background Probabilities: ['0.0001', '0.0269', '0.0000', '0.0035', '0.0035', '0.0050', '0.0023', '0.0007', '0.0107', '0.0006']
Target Probabilities: ['0.9931', '0.9846', '0.9991', '0.9999', '0.9504', '0.9596', '0.9804', '0.9996', '0.9493', '0.9569']


In [8]:
model_path = "training_cls/results/sample_cls/2layer_lr0.0001_bs4/checkpoints/model_epoch_10.pth"
from training_cls.cls_models import CustomLatentClassifier
model = CustomLatentClassifier(input_dim=18*512, num_layers=2, hidden_dim=None).to(device)
# Load weights
print(f"Loaded model from {model_path}")
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()  # Set to evaluation mode
test_model_on_latents(output_latents_1, model, device)

Loaded model from training_cls/results/sample_cls/2layer_lr0.0001_bs4/checkpoints/model_epoch_10.pth
Background Logits: ['-9.0016', '-3.5901', '-10.3554', '-5.6512', '-5.6621', '-5.2857', '-6.0534', '-7.2377', '-4.5239', '-7.4189']
Target Logits: ['4.9717', '4.1590', '7.0104', '9.3728', '2.9523', '3.1670', '3.9125', '7.9452', '2.9300', '3.0998']
Background Probabilities: ['0.0001', '0.0269', '0.0000', '0.0035', '0.0035', '0.0050', '0.0023', '0.0007', '0.0107', '0.0006']
Target Probabilities: ['0.9931', '0.9846', '0.9991', '0.9999', '0.9504', '0.9596', '0.9804', '0.9996', '0.9493', '0.9569']
